# PPR data - load, profile, and data-quality checks

**How to use:** set `DATA_DIR` to the folder with your 5 Excel files, then Run All.
Read the issues table, then set `KEY` (cell 7) from the suggested candidates and
re-run cells 7-8 to check the join and combine into one file.

In [ ]:
import pandas as pd, numpy as np, glob, os, re
from pathlib import Path
from IPython.display import display
pd.set_option('display.max_columns', 200)

# >>> SET THIS to the folder holding your 5 Excel files <<<
DATA_DIR = Path(".")

files = sorted(glob.glob(str(DATA_DIR/"*.xlsx"))) + sorted(glob.glob(str(DATA_DIR/"*.xls")))
print(f"Found {len(files)} Excel files in {DATA_DIR.resolve()}:")
for f in files: print("  -", os.path.basename(f))

In [ ]:
# Load every sheet from every file. Column names trimmed of stray whitespace.
frames = {}
for f in files:
    xl = pd.ExcelFile(f)
    for sh in xl.sheet_names:
        key = os.path.splitext(os.path.basename(f))[0]
        if len(xl.sheet_names) > 1: key += f"::{sh}"
        df = xl.parse(sh)
        df.columns = [str(c).strip() for c in df.columns]
        frames[key] = df
        print(f"{key:50s} rows={len(df):7d}  cols={df.shape[1]}")

In [ ]:
# Profile: dtype, nulls, uniqueness per column
for name, df in frames.items():
    print("="*90); print(name)
    prof = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "nulls":    df.isna().sum(),
        "null_%":   (df.isna().mean()*100).round(1),
        "n_unique": df.nunique(),
    })
    display(prof)

In [ ]:
# Auto-detect the key / center / date columns in each file
def find_cols(df, patterns):
    return [c for c in df.columns if any(re.search(p, c, re.I) for p in patterns)]

ID_PATS   = [r"patient", r"\bid\b", r"pc[-_ ]?form", r"til[-_ ]?odr", r"order.*name"]
CTR_PATS  = [r"\batc\b", r"center", r"veeva", r"account", r"\bhco\b", r"\bname\b"]
DATE_PATS = [r"date", r"dt$"]

keymap = {}
for name, df in frames.items():
    keymap[name] = {"id": find_cols(df, ID_PATS),
                    "center": find_cols(df, CTR_PATS),
                    "date": find_cols(df, DATE_PATS)}
    print(name)
    print("   id     :", keymap[name]["id"])
    print("   center :", keymap[name]["center"])
    print("   date   :", keymap[name]["date"])

In [ ]:
# Data-quality checks: dup rows, null/dup/whitespace in keys, unparseable dates
issues = []
for name, df in frames.items():
    km = keymap[name]
    d = int(df.duplicated().sum())
    if d: issues.append((name, "duplicate full rows", d))
    for idc in km["id"]:
        n = int(df[idc].isna().sum())
        if n: issues.append((name, f"null in id [{idc}]", n))
        if df[idc].dtype == object:
            raw = df[idc].dropna().astype(str)
            w = int((raw != raw.str.strip()).sum())
            if w: issues.append((name, f"whitespace in id [{idc}]", w))
    for cc in km["center"]:
        if df[cc].dtype == object:
            raw = df[cc].dropna().astype(str)
            w = int((raw != raw.str.strip()).sum())
            if w: issues.append((name, f"whitespace in center [{cc}]", w))
            # case-variant duplicates (same name, different casing)
            cv = raw.str.strip().str.upper().nunique()
            if cv < raw.str.strip().nunique():
                issues.append((name, f"case-variant center names [{cc}]", raw.str.strip().nunique()-cv))
    for dc in km["date"]:
        parsed = pd.to_datetime(df[dc], errors="coerce")
        bad = int(parsed.isna().sum() - df[dc].isna().sum())
        if bad > 0: issues.append((name, f"unparseable dates [{dc}]", bad))

iss = pd.DataFrame(issues, columns=["file","issue","count"])
display(iss if len(iss) else "No basic issues found.")

In [ ]:
# Suggest a join key, then check how well IDs overlap across files
from collections import Counter
cand = Counter()
for km in keymap.values():
    for c in km["id"]: cand[c] += 1
print("Candidate join keys (files containing them):", cand.most_common())

# >>> After looking at the candidates above, set KEY to the shared id column <<<
KEY = None   # e.g. "Patient ID"

if KEY:
    sets = {n: set(df[KEY].dropna().astype(str).str.strip())
            for n, df in frames.items() if KEY in df.columns}
    names = list(sets)
    ov = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            ov.loc[a, b] = round(100*len(sets[a] & sets[b]) / max(len(sets[a]), 1), 1)
    print("\n% of row-file's keys also present in column-file (join coverage):")
    display(ov)
else:
    print("Set KEY above from the candidates, then re-run this cell.")

In [ ]:
# Combine into one patient-grain table (only after KEY is set) and export
if KEY:
    tables = [(n, df) for n, df in frames.items() if KEY in df.columns]
    base_name, base = tables[0]
    base = base.copy()
    for n, t in tables[1:]:
        before = len(base)
        base = base.merge(t, on=KEY, how="outer", suffixes=("", f"__{n[:6]}"))
        print(f"merge {n:40s}: {before} -> {len(base)} rows")
    ratio = round(len(base)/base[KEY].nunique(), 2)
    print(f"\nrows per key = {ratio}  (near 1.0 = clean patient grain; >>1 = fan-out, revisit joins)")
    out = DATA_DIR/"combined.xlsx"
    base.to_excel(out, index=False)
    print("wrote", out.resolve())
    display(base.head())
else:
    print("Set KEY first.")